In [81]:
import re

import pyranges as pr
import pandas as pd


paf_file = "/home/ebertp/work/projects/chrom-y-extended/wf-data/seqann/region_db/HG02258.aa2a60f6.hg38-repeat-details.region-db-aln.norm-paf.tsv.gz"

paf = pd.read_csv(paf_file, sep="\t")
paf["pct_matching"] = round(paf["align_matching"] / paf["query_length"] * 100, 2)

def get_max_identity_block(cigar_string):

    max_block = 0
    for block in re.finditer("[0-9]+\=", cigar_string):
        max_block = max(max_block, int(block.group(0)[:-1]))
    return max_block


paf["max_id_block"] = paf["cg_cigar"].apply(get_max_identity_block)

paf["rank_matching"] = paf["pct_matching"].rank(method="average", ascending=True, pct=True)
paf["rank_block"] = paf["max_id_block"].rank(method="average", ascending=True, pct=True)
paf["mean_rank"] = ((paf.rank_matching + paf.rank_block) / 2 * 1000).round(0).astype(int)
paf["strand"] = paf.align_orient.replace({1: "+", -1: "-"}).astype(str)

target_iv = pr.from_dict(
    {
        "Chromosome": paf.target_name,
        "Start": paf.target_start,
        "End": paf.target_end,
        "Strand": paf.strand,
        "Name": paf.query_name,
        "pd_idx": paf.index.values
    }
)

target_iv = target_iv.cluster(strand="same", by="Name")
cluster_ids = pd.Series(
    target_iv.Cluster.values,
    index=target_iv.pd_idx.values,
    name="cluster_id"
)

# build in a sanity check given the dev stage of PyRanges
_n_rows = paf.shape[0]

paf = paf.merge(cluster_ids, left_index=True, right_index=True)

assert paf.shape[0] == _n_rows

bed_rows = []
for cluster_id, alignments in paf.groupby("cluster_id"):
    assert alignments.query_name.nunique() == 1
    seq = alignments.target_name.iloc[0]
    name = alignments.query_name.iloc[0]
    strand = alignments.strand.iloc[0]
    start = alignments.target_start.min()
    end = alignments.target_end.max()
    assert start < end
    score = alignments.mean_rank.max()
    max_pct = alignments.pct_matching.max()
    max_block = alignments.max_id_block.max()
    new_label = f"{name}[IDPCT:{max_pct}|IDBLK:{max_block}]"
    bed_rows.append(
        (seq, start, end, new_label, score, strand, name, max_pct, max_block, alignments.shape[0])
    )

bed_df = pd.DataFrame.from_records(
    bed_rows,
    columns=[
        "chrom", "start", "end", "name", "score", "strand",
        "label", "max_pct_align_match", "longest_id_block",
        "merged_alignments"
    ],
)

bed_df.sort_values(["chrom", "start", "end"], inplace=True)

print(bed_df.head(10))

            chrom    start      end  \
0    HG02258_chrY     3171   105289   
139  HG02258_chrY    20184    33479   
1    HG02258_chrY   105508   106157   
2    HG02258_chrY   110698   149995   
3    HG02258_chrY   150411   211975   
4    HG02258_chrY   213519   213836   
5    HG02258_chrY   214048   709508   
6    HG02258_chrY   710432  2417814   
7    HG02258_chrY  2417814  2686351   
8    HG02258_chrY  2686351  5874209   

                                            name  score strand  \
0     chrY_hg38_01n_PAR1[IDPCT:3.46|IDBLK:10647]    430      +   
139     chrY_hg38_01n_PAR1[IDPCT:0.45|IDBLK:403]     86      -   
1       chrY_hg38_01n_PAR1[IDPCT:0.01|IDBLK:143]     36      +   
2     chrY_hg38_01n_PAR1[IDPCT:1.38|IDBLK:12413]    438      +   
3      chrY_hg38_01n_PAR1[IDPCT:2.16|IDBLK:5598]    298      +   
4       chrY_hg38_01n_PAR1[IDPCT:0.01|IDBLK:304]     42      +   
5      chrY_hg38_01n_PAR1[IDPCT:16.7|IDBLK:7500]    392      +   
6    chrY_hg38_01n_PAR1[IDPCT:36.77|IDBLK: